In [18]:
from llama_index.core.indices.multi_modal.base import MultiModalVectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import SimpleDirectoryReader, StorageContext
from llama_index.embeddings.clip import ClipEmbedding
from llama_index.embeddings.cohere import CohereEmbedding
from dotenv import load_dotenv
import os 

In [19]:
_ = load_dotenv(".env")

### Load images and text documents

In [ ]:
def load_documents():
    """Load the context images and text into ImageDocument and Documents"""
    # context images
    image_path = "../datasets_chatbot/images"
    image_documents = SimpleDirectoryReader(image_path).load_data()
    

    # context text
    text_path = "../datasets_chatbot/descriptions"
    text_documents = SimpleDirectoryReader(text_path).load_data()

    return image_documents, text_documents

In [6]:
image_documents, text_documents = load_documents()

In [8]:
print(image_documents)
print(len(image_documents))
print(text_documents)
print(len(text_documents))

[ImageDocument(id_='c47ffa64-aee1-439d-9b42-3efb7cff9a1e', embedding=None, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\images\\14.jpg', 'file_name': '14.jpg', 'file_type': 'image/jpeg', 'file_size': 171654, 'creation_date': '2024-11-14', 'last_modified_date': '2024-10-26'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, text='', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_template='{key}: {value}', metadata_seperator='\n', image=None, image_path='d:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\images\\14.jpg', image_url=None, image_mimetype=None, text_embedding=None), ImageDo

### Create Multimodal VectorStore Index

In [10]:
# Define the sentence splitter
node_parser = SentenceSplitter.from_defaults()

In [14]:
image_nodes = node_parser.get_nodes_from_documents(image_documents)
print(image_nodes)
print(image_nodes[0].metadata['file_path'])
print(image_nodes[1].metadata['file_path'])
print(image_nodes[2].metadata['file_path'])
print(len(image_nodes))

[ImageNode(id_='5834a1ce-0834-45a3-a939-e121c78dcb60', embedding=None, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\images\\14.jpg', 'file_name': '14.jpg', 'file_type': 'image/jpeg', 'file_size': 171654, 'creation_date': '2024-11-14', 'last_modified_date': '2024-10-26'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='c47ffa64-aee1-439d-9b42-3efb7cff9a1e', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\images\\14.jpg', 'file_name': '14.jpg', 'file_type': 'image/jpeg', 'file_size': 171654, 'creation_date': '2024-11-14', 'last_modified_

In [16]:
text_nodes = node_parser.get_nodes_from_documents(text_documents)
print(text_nodes)
print(text_nodes[0].metadata['file_path'])
print(text_nodes[1].metadata['file_path'])
print(text_nodes[2].metadata['file_path'])
print(len(text_nodes))

[TextNode(id_='49609130-595c-40e1-9da2-9795d46c4ac6', embedding=None, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\14.txt', 'file_name': '14.txt', 'file_type': 'text/plain', 'file_size': 1047, 'creation_date': '2024-11-14', 'last_modified_date': '2024-11-13'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='a0f48177-3244-41d2-a47a-0d2c051060e5', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'd:\\workspace\\proyecto_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\14.txt', 'file_name': '14.txt', 'file_type': 'text/plain', 'file_size': 1047, 'creation_date': '2024-11-14', 'last_mo

In [20]:
# Create Cohere embedding 
embed_model = CohereEmbedding(
    api_key=os.getenv('COHERE_API_KEY')
)

In [21]:
image_embed_model = ClipEmbedding()

In [22]:
# Create multimodal index
index = MultiModalVectorStoreIndex(
    nodes = image_nodes + text_nodes,
    image_embed_model=image_embed_model,
    embed_model = embed_model
)

In [23]:
index

In [39]:
def create_multimodal_index(image_documents, text_documents):
    # Create Cohere embedding 
    embed_model = CohereEmbedding(api_key=os.getenv('COHERE_API_KEY'))

    # cretate image embed model
    image_embed_model = ClipEmbedding()

    # Define the sentence splitter
    node_parser = SentenceSplitter.from_defaults()
    image_nodes = node_parser.get_nodes_from_documents(image_documents)
    text_nodes = node_parser.get_nodes_from_documents(text_documents)

    # Create multimodal index
    index = MultiModalVectorStoreIndex(
    nodes = image_nodes + text_nodes,
    image_embed_model=image_embed_model,
    embed_model = embed_model
    )

    return index
